In [1]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
def search(query: str) -> str:
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
search('How do I run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [4]:
import os
from rag_helper import RAGBase
from dotenv import load_dotenv
from google import genai

# openai_client = OpenAI()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

load_dotenv()


True

In [5]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [6]:
from google.genai import types

# 2. Package your manual schema design 
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search",
            description="Search the FAQ database for entries matching the given query.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(
                        type=types.Type.STRING,
                        description="Search query text to look up in the course FAQ."
                    )
                },
                required=["query"]
            )
        )
    ]
)


# 3. Add BOTH the schema tool wrapper AND the python function reference into tools
config = types.GenerateContentConfig(
    # The SDK maps the schema name to the matching function name in this list
    tools=[search], 
    system_instruction=instructions,
    temperature=0.7
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=config
)

# Gemini will see the prompt, decide to call 'search', execute your 
# local python function, receive the database result, and give you the final text answer.
response = chat.send_message("I just discovered the llm zoomcamp course. Can I join it?")
print(response.text)

Yes, you can still join the LLM Zoomcamp course! However, if you wish to receive a certificate, you'll need to submit your project while submissions are still being accepted.

You can track the course syllabus, deadlines, homework, and your progress on the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

You don't necessarily need to register to start learning and submitting homework. Registration is primarily used to gauge interest before the course begins.

Is there anything else you'd like to know about the course?


In [8]:
chat.get_history()

[UserContent(
   parts=[
     Part(
       text='I just discovered the llm zoomcamp course. Can I join it?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'llm zoomcamp course join'
         },
         name='search'
       ),
       thought_signature=b'\n\x97\x02\x01\x0c9\xd6\xc7\xc4\xab\x15\x97\xe1\xea7\xdb}G\x8e\x9b\xa2\xfe\x9b\xb6z{V\xe7m9P\xa3n\xb7\x1c\xb8LK\xd3\xad\xd58\x81\x8a\xa0\xc3X\xfeK\xddZ[\x9cL\xd2\xe2L\xbd9\xc0\x9c\x8fT\xff|#r\xff\xe8\x1c6\xa5\xbcq&\xd7\xdf\xddm\xad\xc1f\xc4\xcfx\x02\xc1u\x83\x7f\xee\x02eI\x13\xb1\xb9...'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='search',
         response={
           'result': [
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
    

In [9]:
# Dig into the parts list first, then fetch the text from the first part
final_text = chat.get_history()[-1].parts[0].text

print("\n--- Actual Final Response ---")
print(final_text)


--- Actual Final Response ---
Yes, you can still join the LLM Zoomcamp course! However, if you wish to receive a certificate, you'll need to submit your project while submissions are still being accepted.

You can track the course syllabus, deadlines, homework, and your progress on the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

You don't necessarily need to register to start learning and submitting homework. Registration is primarily used to gauge interest before the course begins.

Is there anything else you'd like to know about the course?
